In [56]:
import zipfile
import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


zip_path = '/content/Resume.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/resumes_data')


!python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

print("Resumes unzipped and spaCy model loaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 46.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Resumes unzipped and spaCy model loaded.


In [55]:

resume_csv_path = '/content/resumes_data/Resume.csv'
resume_df = pd.read_csv(resume_csv_path)


if 'Resume_str' in resume_df.columns:
    resume_df = resume_df.rename(columns={'Resume_str': 'content'})
elif 'Resume_text' in resume_df.columns:
    resume_df = resume_df.rename(columns={'Resume_text': 'content'})


resume_df = resume_df.head(100).copy()

display(resume_df.head())

,ID,content,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [54]:
job_description = """
We are looking for a Data Scientist with experience in Python, Scikit-learn, and NLP.
The candidate should be able to build machine learning models, perform text analysis,
and have experience with data visualization tools like Matplotlib or Seaborn.
"""


full_df = pd.read_csv('/content/resumes_data/Resume.csv')


if 'Resume_str' in full_df.columns:
    full_df = full_df.rename(columns={'Resume_str': 'content'})


target_category = 'INFORMATION-TECHNOLOGY'
if not full_df[full_df['Category'].str.contains('Data Science', case=False, na=False)].empty:
    target_category = 'Data Science'

ds_df = full_df[full_df['Category'].str.contains(target_category, case=False, na=False)].copy()

def rank_resumes(job_desc, df):
    if df.empty:
        return []
    documents = [job_desc] + df['content'].astype(str).tolist()
    vectorizer = TfidfVectorizer(stop_words='english').fit_transform(documents)
    vectors = vectorizer.toarray()
    job_desc_vector = vectors[0].reshape(1, -1)
    resume_vectors = vectors[1:]
    similarities = cosine_similarity(job_desc_vector, resume_vectors)[0]
    return similarities


if not ds_df.empty:
    ds_df['fit_score'] = rank_resumes(job_description, ds_df)
    resume_df = ds_df.sort_values(by='fit_score', ascending=False)
    print(f"Top Ranked Candidates in Category: {target_category}")
    display(resume_df[['ID', 'Category', 'fit_score']].head())
else:
    print(f"Category {target_category} not found.")

Top Ranked Candidates in Category: INFORMATION-TECHNOLOGY


,ID,Category,fit_score
297,83816738,INFORMATION-TECHNOLOGY,0.082421
331,18067556,INFORMATION-TECHNOLOGY,0.072330
253,22450718,INFORMATION-TECHNOLOGY,0.065426
220,25990239,INFORMATION-TECHNOLOGY,0.059176
315,10265057,INFORMATION-TECHNOLOGY,0.054326


In [53]:

required_skills = ['python', 'scikit-learn', 'nlp', 'matplotlib', 'seaborn', 'machine learning', 'text analysis']

def extract_skills(text):
    doc = nlp(text.lower())

    found_skills = set()
    for token in doc:
        if token.text in required_skills:
            found_skills.add(token.text)

    for skill in ['machine learning', 'text analysis', 'scikit-learn']:
        if skill in text.lower():
            found_skills.add(skill)
    return list(found_skills)


top_candidates = resume_df.head(5).copy()
top_candidates['matched_skills'] = top_candidates['content'].apply(extract_skills)
top_candidates['missing_skills'] = top_candidates['matched_skills'].apply(lambda x: [s for s in required_skills if s not in x])

print("Skill Analysis for Top Candidates:")
display(top_candidates[['ID', 'Category', 'fit_score', 'matched_skills', 'missing_skills']])

Skill Analysis for Top Candidates:


,ID,Category,fit_score,matched_skills,missing_skills
297,83816738,INFORMATION-TECHNOLOGY,0.082421,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
331,18067556,INFORMATION-TECHNOLOGY,0.072330,[python],"[scikit-learn, nlp, matplotlib, seaborn, machi..."
253,22450718,INFORMATION-TECHNOLOGY,0.065426,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
220,25990239,INFORMATION-TECHNOLOGY,0.059176,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
315,10265057,INFORMATION-TECHNOLOGY,0.054326,[python],"[scikit-learn, nlp, matplotlib, seaborn, machi..."


In [52]:
import re

required_skills = ['python', 'scikit-learn', 'nlp', 'matplotlib', 'seaborn', 'machine learning', 'sql', 'statistics', 'pandas']

def extract_skills(text):
    text_lower = str(text).lower()
    found_skills = set()
    for skill in required_skills:

        pattern = r'\\b' + re.escape(skill).replace('-', '[- ]?') + r'\\b'
        if re.search(pattern, text_lower):
            found_skills.add(skill)
    return list(found_skills)


if 'resume_df' in locals() and not resume_df.empty and 'fit_score' in resume_df.columns:
    top_candidates = resume_df.head(5).copy()
    top_candidates['matched_skills'] = top_candidates['content'].apply(extract_skills)
    top_candidates['missing_skills'] = top_candidates['matched_skills'].apply(lambda x: [s for s in required_skills if s not in x])

    print('Detailed Skill Analysis for Top Technical Candidates:')
    display(top_candidates[['ID', 'fit_score', 'matched_skills', 'missing_skills']])
else:
    print('Ranking failed or fit_score not found. Please re-run the ranking cell (c09a0d6b).')


Detailed Skill Analysis for Top Technical Candidates:


,ID,fit_score,matched_skills,missing_skills
297,83816738,0.082421,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
331,18067556,0.072330,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
253,22450718,0.065426,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
220,25990239,0.059176,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
315,10265057,0.054326,[],"[python, scikit-learn, nlp, matplotlib, seabor..."
